In [1]:
%load_ext autoreload
%autoreload 2
import dask
import dask.distributed
from dask_util import DaskClient
import dask_util
import numpy as np

In [2]:
local_params = {
    "n_workers":4, 
    "processes" : True, 
    "dashboard_address" : 'localhost:7777'
    
}

# cluster = {
#     "cores" : 24,
#     "processes" : 1,
#     "memory" : "1GB",
#     "shebang" : '#!/usr/bin/env bash',
#     "queue" : "serc",
#     "walltime" : "00:10:00",
#     "local_directory" : '/tmp',
#     "death_timeout" : "15s",
#     "interface" : "ib0",
#     "log_directory" : f'{os.environ["SCRATCH"]}/dask_jobqueue_logs/'    
# }


client = DaskClient(local_params=local_params)

2023-03-09 23:21:42,864 - distributed.diskutils - INFO - Found stale lock file and directory '/tmp/dask-worker-space/worker-xlht79wb', purging


In [3]:
%load_ext autoreload
%autoreload 2
import SepVector
from __pyDaskVector import DaskVector
from __pyDaskOperator import DaskOperator
import Hypercube
import pyOperator as Op


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
WARNING! DATAPATH not found. The folder /tmp will be used to write binary files


/usr/local/lib/python3.9/dist-packages/numpy/core/getlimits.py:500: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/lib/python3.9/dist-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/usr/local/lib/python3.9/dist-packages/numpy/core/getlimits.py:500: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/lib/python3.9/dist-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)


In [4]:

ns = [10,4]
os = [0,0]
ds = [1,1]
chunks = (1,3)

ax = Hypercube.axis(n=1, o=1, d=1)
hyp = Hypercube.hypercube(ns=ns, ds=ds, os=os)
vec = SepVector.getSepVector(ns=ns, ds=ds, os=os)
vec.set(1)

floatVector
Axis 1: n=10	o=0.000000	d=1.000000
Axis 2: n=4	o=0.000000	d=1.000000

In [5]:
# option 1
# creating from scratch
data = DaskVector(client, vecCls=SepVector.floatVector, ns=ns, ds=ds, os=os, chunks=chunks)

In [6]:
data[:]

[array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]], dtype=float32),
 array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]], dtype=float32),
 array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]], dtype=float32)]

In [7]:
# option 2
# creating from existing in-memory SepVector
daskVec = DaskVector(client, from_vector=vec, chunks=chunks)

In [8]:
d2 = data.clone()

In [9]:
client.getClient().has_what()

{'tcp://127.0.0.1:36497': ('floatVector-23ac126dfc94fac02130f857ddb0463c',
  'clone-8b10a784-62ad-445e-9656-97fdedf0800a-0'),
 'tcp://127.0.0.1:40309': ('floatVector-8d5594ad1f43a85add8ca7e33ee97699',
  'clone-8b10a784-62ad-445e-9656-97fdedf0800a-1'),
 'tcp://127.0.0.1:40681': ('floatVector-23d4f692b5f73d4c6c876af3426f9c54',
  'window-5df18b876d4224453d0e44fff2377022',
  'clone-8b10a784-62ad-445e-9656-97fdedf0800a-2'),
 'tcp://127.0.0.1:42681': ('window-86bf330356a4a0c72327d1185d1c9a0b',
  'window-a15a8eda572d0f9f42557d45ab1ff54c')}

In [10]:
daskVec.checkSame(daskVec)

True

In [11]:
daskVec.getHyper()

Axis 1: n=10	o=0.000000	d=1.000000
Axis 2: n=4	o=0.000000	d=1.000000

In [13]:
scaleOp = DaskOperator(client, Op.scalingOp, daskVec, data, 4)

In [14]:
scaleOp.forward(False, daskVec, data)

[[<Future: pending, key: fwd-2826f2f8-40bc-4ca4-a574-29e3b6f285e2-0>, <Future: pending, key: fwd-2826f2f8-40bc-4ca4-a574-29e3b6f285e2-1>, <Future: pending, key: fwd-2826f2f8-40bc-4ca4-a574-29e3b6f285e2-2>], [<Future: pending, key: fwd-8bab4dec-aa67-4062-99d6-e97b2482b6ce-0>, <Future: pending, key: fwd-8bab4dec-aa67-4062-99d6-e97b2482b6ce-1>, <Future: pending, key: fwd-8bab4dec-aa67-4062-99d6-e97b2482b6ce-2>], [<Future: pending, key: fwd-a7418fc1-3eb5-4556-93ae-7ff961a8c46d-0>, <Future: pending, key: fwd-a7418fc1-3eb5-4556-93ae-7ff961a8c46d-1>, <Future: pending, key: fwd-a7418fc1-3eb5-4556-93ae-7ff961a8c46d-2>]]


/usr/local/lib/python3.9/dist-packages/numpy/core/getlimits.py:500: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/lib/python3.9/dist-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/usr/local/lib/python3.9/dist-packages/numpy/core/getlimits.py:500: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/lib/python3.9/dist-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/usr/local/lib/python3.9/dist-packages/numpy/core/getlimits.py:500: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is z

In [15]:
data[:]

[array([[4., 4., 4., 4., 4., 4., 4., 4., 4., 4.]], dtype=float32),
 array([[4., 4., 4., 4., 4., 4., 4., 4., 4., 4.]], dtype=float32),
 array([[4., 4., 4., 4., 4., 4., 4., 4., 4., 4.],
        [4., 4., 4., 4., 4., 4., 4., 4., 4., 4.]], dtype=float32)]

In [ ]:
scaleOp.adjoint(False, daskVec, data)

In [ ]:
daskVec[:]